# MEDIAPIPE (GOOGLE) - UNO SGUARDO AL FUTURO

# IL MODULO FACE - LANDMARKER

In [ ]:
!pip install mediapipe
!wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
print("✅ Modello scaricato con successo!")

# TEST VIDEOCAMERA

In [ ]:
from IPython.display import display, Javascript

test_code = '''
    async function testCamera() {
        try {
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            alert("✅ Webcam funzionante! I permessi del browser sono corretti.");
            // Spegne subito la videocamera dopo il test
            stream.getTracks().forEach(track => track.stop());
        } catch (err) {
            alert("❌ Errore Webcam bloccata: " + err.message);
        }
    }
    testCamera();
'''
display(Javascript(test_code))

# RILEVAZIONE DEI PUNTI CARATTERISTICI DEL VISO

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from google.colab.patches import cv2_imshow

# 1. Funzione JavaScript per scattare una foto
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '🔴 CLICCA QUI PER SCATTARE (CAPTURE)';
      capture.style.padding = '10px';
      capture.style.fontSize = '16px';
      capture.style.backgroundColor = 'red';
      capture.style.color = 'white';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Aspetta che l'utente clicchi il pulsante
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# 2. Inizializzazione del "Cervello" AI (Tasks API)
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(base_options=base_options, num_faces=1)
detector = vision.FaceLandmarker.create_from_options(options)

# 3. Esecuzione del Programma
try:
  print("🚀 Avvio sistemi telecamera... Clicca sul pulsante rosso per acquisire la telemetria!")
  filename = take_photo()
  print("✅ Immagine acquisita! Elaborazione in corso...")

  # Lettura immagine con OpenCV
  img = cv2.imread(filename)
  img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)

  # Rilevamento dei punti
  detection_result = detector.detect(mp_image)

  if detection_result.face_landmarks:
      for face_landmarks in detection_result.face_landmarks:

          # --- CICLO FOR: Disegna la griglia ---
          for landmark in face_landmarks:
              x_px = int(landmark.x * img.shape[1])
              y_px = int(landmark.y * img.shape[0])
              cv2.circle(img, (x_px, y_px), 1, (0, 255, 0), -1)

          # --- LOGICA IF: Calcolo Assetto di Volo ---
          # Estraiamo il punto 1 (punta del naso)
          naso = face_landmarks[1]
          x_naso = int(naso.x * img.shape[1])
          y_naso = int(naso.y * img.shape[0])

          # Evidenziamo il naso in rosso
          cv2.circle(img, (x_naso, y_naso), 6, (0, 0, 255), -1)

          # Decisione basata sulla posizione X del naso
          if naso.x < 0.45:
              testo = "SPOSTAMENTO A DESTRA"
              colore = (0, 0, 255) # Rosso
          elif naso.x > 0.55:
              testo = "SPOSTAMENTO A SINISTRA"
              colore = (0, 0, 255) # Rosso
          else:
              testo = "ASSETTO STABILE"
              colore = (0, 255, 255) # Giallo

          # Stampiamo il testo sull'immagine
          cv2.putText(img, testo, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, colore, 2)
  else:
      print("❌ Nessuna persona rilevata nell'inquadratura.")

  # Mostra il risultato finale agli studenti!
  cv2_imshow(img)

except Exception as err:
  print(f"Errore: {str(err)}")

# RILEVAZIONE DINAMICA

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import display, Javascript
import google.colab.output
from base64 import b64decode, b64encode

# 1. Inizializziamo il modello AI
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(base_options=base_options, num_faces=1)
detector = vision.FaceLandmarker.create_from_options(options)

# 2. La funzione Python che elabora il singolo fotogramma
def process_frame(data_url):
    # Decodifica l'immagine in arrivo dal browser
    image_bytes = b64decode(data_url.split(',')[1])
    img_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(img_np, flags=1)

    # Preparazione per MediaPipe
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)

    # Rilevamento
    detection_result = detector.detect(mp_image)

    if detection_result.face_landmarks:
        face_landmarks = detection_result.face_landmarks[0]

        # --- CICLO FOR: Disegna la maschera ---
        for landmark in face_landmarks:
            x_px = int(landmark.x * img.shape[1])
            y_px = int(landmark.y * img.shape[0])
            cv2.circle(img, (x_px, y_px), 1, (0, 255, 0), -1)

        # --- LOGICA IF: Sensori di Navigazione ---
        naso = face_landmarks[1]
        x_naso = int(naso.x * img.shape[1])
        y_naso = int(naso.y * img.shape[0])
        cv2.circle(img, (x_naso, y_naso), 6, (0, 0, 255), -1)

        if naso.x < 0.45:
            cv2.putText(img, "SPOSTAMENTO A DESTRA", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        elif naso.x > 0.55:
            cv2.putText(img, "SPOSTAMENTO A SINITRA SINISTRA", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        else:
            cv2.putText(img, "ASSETTO STABILE", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

    # Ricodifica l'immagine per rimandarla allo schermo dello studente
    _, buffer = cv2.imencode('.jpg', img)
    return "data:image/jpeg;base64," + b64encode(buffer).decode('utf-8')

# Registriamo la funzione per farla chiamare da JavaScript
google.colab.output.register_callback('notebook.process_frame', process_frame)

# 3. JavaScript: Il Motore del Flusso Continuo
js_code = '''
async function startStream() {
    const div = document.createElement('div');

    // Bottone di sicurezza
    const stopBtn = document.createElement('button');
    stopBtn.textContent = '⏹ FERMA MOTORI (STOP VIDEO)';
    stopBtn.style.padding = '10px';
    stopBtn.style.background = 'red';
    stopBtn.style.color = 'white';
    stopBtn.style.marginBottom = '10px';
    div.appendChild(stopBtn);

    const video = document.createElement('video');
    video.style.display = 'none'; // Nascondiamo il feed grezzo

    const img = document.createElement('img'); // Qui mostriamo l'HUD
    img.style.display = 'block';

    div.appendChild(video);
    div.appendChild(img);
    document.body.appendChild(div);

    const stream = await navigator.mediaDevices.getUserMedia({video: true});
    video.srcObject = stream;
    await video.play();

    const canvas = document.createElement('canvas');
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    const ctx = canvas.getContext('2d');

    let isPlaying = true;

    // Condizione di uscita dal ciclo
    stopBtn.onclick = () => {
        isPlaying = false;
        stream.getTracks().forEach(track => track.stop());
        div.remove();
        console.log("Sistemi disattivati.");
    };

    // Il Ciclo Continuo (Loop di Telemetria)
    while (isPlaying) {
        ctx.drawImage(video, 0, 0);
        const dataUrl = canvas.toDataURL('image/jpeg', 0.6); // Compressione per non intasare la rete

        try {
            const result = await google.colab.kernel.invokeFunction('notebook.process_frame', [dataUrl], {});
            img.src = result.data['text/plain'].replace(/'/g, "");
        } catch (e) {
            console.error("Errore di trasmissione:", e);
        }

        // Pausa di 60ms (circa 15 FPS) per dare respiro al server
        await new Promise(r => setTimeout(r, 60));
    }
}
startStream();
'''
display(Javascript(js_code))

# ESERCIZIO: MODIFICARE IL CODICE PER RILEVARE IL BATTITO DEGLI OCCHI, PER RILEVARE UN SORRISO, PER RILEVARE UN OCCHIOLINO

UTILIZZATE LA VOSTRA CREATIVITA'

# IL MODULO HAND-LANDMARKER

In [ ]:
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
print("✅ Modello di tracciamento MANO scaricato e pronto all'uso!")

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import display, Javascript
import google.colab.output
from base64 import b64decode, b64encode

# 1. Inizializziamo il modello AI per le MANI
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)

# 2. La funzione Python che elabora i fotogrammi
def process_hand_frame(data_url):
    image_bytes = b64decode(data_url.split(',')[1])
    img_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(img_np, flags=1)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)

    # Rilevamento della mano
    detection_result = detector.detect(mp_image)

    if detection_result.hand_landmarks:
        # Prendiamo la prima mano rilevata
        hand_landmarks = detection_result.hand_landmarks[0]

        # --- CICLO FOR: Disegna lo scheletro della mano ---
        for landmark in hand_landmarks:
            x_px = int(landmark.x * img.shape[1])
            y_px = int(landmark.y * img.shape[0])
            cv2.circle(img, (x_px, y_px), 4, (255, 0, 0), -1) # Punti blu

        # --- LOGICA IF: Manovra del braccio robotico ---
        # Il punto 8 è la PUNTA DELL'INDICE
        indice = hand_landmarks[8]
        x_ind = int(indice.x * img.shape[1])
        y_ind = int(indice.y * img.shape[0])

        # Evidenziamo l'indice con un cerchio più grande
        cv2.circle(img, (x_ind, y_ind), 10, (0, 255, 255), -1)

        # Controlli direzionali sull'asse X
        if indice.x < 0.35:
            testo = "BRACCIO: SPOSTAMENTO A SINISTRA"
            colore = (0, 255, 0)
        elif indice.x > 0.65:
            testo = "BRACCIO: SPOSTAMENTO A DESTRA"
            colore = (0, 255, 0)
        else:
            testo = "BRACCIO: POSIZIONE CENTRALE"
            colore = (255, 255, 255)

        cv2.putText(img, testo, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, colore, 2)

    _, buffer = cv2.imencode('.jpg', img)
    return "data:image/jpeg;base64," + b64encode(buffer).decode('utf-8')

google.colab.output.register_callback('notebook.process_hand_frame', process_hand_frame)

# 3. JavaScript: Il Motore del Flusso Continuo
js_code = '''
async function startHandStream() {
    const div = document.createElement('div');
    const stopBtn = document.createElement('button');
    stopBtn.textContent = '⏹ FERMA SISTEMA ROBOTICO';
    stopBtn.style.padding = '10px';
    stopBtn.style.background = 'red';
    stopBtn.style.color = 'white';
    stopBtn.style.marginBottom = '10px';
    div.appendChild(stopBtn);

    const video = document.createElement('video');
    video.style.display = 'none';
    const img = document.createElement('img');
    img.style.display = 'block';

    div.appendChild(video);
    div.appendChild(img);
    document.body.appendChild(div);

    const stream = await navigator.mediaDevices.getUserMedia({video: true});
    video.srcObject = stream;
    await video.play();

    const canvas = document.createElement('canvas');
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    const ctx = canvas.getContext('2d');

    let isPlaying = true;

    stopBtn.onclick = () => {
        isPlaying = false;
        stream.getTracks().forEach(track => track.stop());
        div.remove();
        console.log("Sistema disattivato.");
    };

    while (isPlaying) {
        ctx.drawImage(video, 0, 0);
        const dataUrl = canvas.toDataURL('image/jpeg', 0.6);

        try {
            const result = await google.colab.kernel.invokeFunction('notebook.process_hand_frame', [dataUrl], {});
            img.src = result.data['text/plain'].replace(/'/g, "");
        } catch (e) {}

        await new Promise(r => setTimeout(r, 60));
    }
}
startHandStream();
'''
display(Javascript(js_code))

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import display, Javascript
import google.colab.output
from base64 import b64decode, b64encode
import math # Necessario per il calcolo della distanza

# 1. Inizializziamo il modello AI
# Assicurati di aver caricato il file 'hand_landmarker.task' nei file di Colab
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)

def process_hand_frame(data_url):
    image_bytes = b64decode(data_url.split(',')[1])
    img_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(img_np, flags=1)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    detection_result = detector.detect(mp_image)

    if detection_result.hand_landmarks:
        hand_landmarks = detection_result.hand_landmarks[0]

        # Disegno dello scheletro
        for landmark in hand_landmarks:
            x_px = int(landmark.x * img.shape[1])
            y_px = int(landmark.y * img.shape[0])
            cv2.circle(img, (x_px, y_px), 3, (255, 0, 0), -1)

        # --- LOGICA: RILEVAMENTO TOCCO INDICE-POLLICE ---
        pollice = hand_landmarks[4]
        indice = hand_landmarks[8]

        # Calcolo distanza euclidea 3D tra punta indice e punta pollice
        distanza_dita = math.sqrt(
            (pollice.x - indice.x)**2 +
            (pollice.y - indice.y)**2 +
            (pollice.z - indice.z)**2
        )

        # Se la distanza è inferiore a una soglia (0.05 è un buon valore standard)
        if distanza_dita < 0.05:
            testo_tocco = "ATTENZIONE: INDICE TOCCA IL POLLICE!"
            # Disegniamo un cerchio di conferma tra le due dita
            cx = int((pollice.x + indice.x) / 2 * img.shape[1])
            cy = int((pollice.y + indice.y) / 2 * img.shape[0])
            cv2.circle(img, (cx, cy), 15, (0, 0, 255), 2)
            cv2.putText(img, testo_tocco, (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # --- LOGICA: MOVIMENTO BRACCIO ---
        if indice.x < 0.35:
            testo = "BRACCIO: SINISTRA"
            colore = (0, 255, 0)
        elif indice.x > 0.65:
            testo = "BRACCIO: DESTRA"
            colore = (0, 255, 0)
        else:
            testo = "BRACCIO: CENTRALE"
            colore = (255, 255, 255)

        cv2.putText(img, testo, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, colore, 2)

    _, buffer = cv2.imencode('.jpg', img)
    return "data:image/jpeg;base64," + b64encode(buffer).decode('utf-8')

# Registrazione callback e JavaScript rimangono invariati
google.colab.output.register_callback('notebook.process_hand_frame', process_hand_frame)

js_code = '''
async function startHandStream() {
    const div = document.createElement('div');
    const stopBtn = document.createElement('button');
    stopBtn.textContent = '⏹ FERMA SISTEMA ROBOTICO';
    stopBtn.style.cssText = 'padding:10px; background:red; color:white; margin-bottom:10px; cursor:pointer;';
    div.appendChild(stopBtn);

    const video = document.createElement('video');
    video.style.display = 'none';
    const img = document.createElement('img');
    img.style.display = 'block';

    div.appendChild(video);
    div.appendChild(img);
    document.body.appendChild(div);

    const stream = await navigator.mediaDevices.getUserMedia({video: true});
    video.srcObject = stream;
    await video.play();

    const canvas = document.createElement('canvas');
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    const ctx = canvas.getContext('2d');

    let isPlaying = true;
    stopBtn.onclick = () => {
        isPlaying = false;
        stream.getTracks().forEach(track => track.stop());
        div.remove();
    };

    while (isPlaying) {
        ctx.drawImage(video, 0, 0);
        const dataUrl = canvas.toDataURL('image/jpeg', 0.5);
        try {
            const result = await google.colab.kernel.invokeFunction('notebook.process_hand_frame', [dataUrl], {});
            img.src = result.data['text/plain'].replace(/'/g, "");
        } catch (e) {}
        await new Promise(r => setTimeout(r, 50));
    }
}
startHandStream();
'''
display(Javascript(js_code))